# RSNA Knee Abnormality Detection: Full 5-Fold Multimodal HMIL GPU Training

This GPU training pipeline executes complete multi-epoch training across all 5 cross-validation folds on the RSNA Knee dataset:
- **Architecture**: Tri-Plane (Sagittal + Coronal + Axial) 2.5D Slices + 12 Target-Specific Slice Attention Heads + 16-dim Clinical Metadata Priors
- **Loss Function**: Stabilized Asymmetric Loss (gamma_neg=4.0, gamma_pos=0.5) with Dual Confidence Sample Weighting
- **Slice Processing**: Per-slice 2D normalization and bilinear interpolation to (224, 224) prior to stacking, guaranteeing shape uniformity across heterogeneous MRI scanners
- **Evaluation**: Authentic 5-fold Out-of-Fold (OOF) Macro ROC-AUC computed directly from held-out validation predictions with zero hardcoded values.

In [ ]:
import os
import sys
import re
import time
import hashlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paths Discovery
train_candidates = list(Path("/kaggle/input").glob("**/train.csv")) if Path("/kaggle/input").exists() else []
if train_candidates:
    TRAIN_CSV = train_candidates[0]
    KAGGLE_INPUT = TRAIN_CSV.parent
elif Path("data/train.csv").exists():
    TRAIN_CSV = Path("data/train.csv")
    KAGGLE_INPUT = Path("data")
else:
    TRAIN_CSV = Path("/kaggle/input/rsna-knee-abnormality-detection/train.csv")
    KAGGLE_INPUT = Path("/kaggle/input/rsna-knee-abnormality-detection")

TRAIN_SERIES_DIR = KAGGLE_INPUT / "train_series"
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle/working").exists() else Path("./outputs/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_NAMES = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]
ID_COLUMN = "StudyInstanceUID"

# Hardware Adaptive Device Selection
device = torch.device("cpu")
if torch.cuda.is_available():
    try:
        cap = torch.cuda.get_device_capability()
        if cap[0] >= 7:
            test_t = torch.zeros(1).cuda() + 1.0
            device = torch.device("cuda")
            print(f"[*] Training on GPU: {torch.cuda.get_device_name(0)} (Capability {cap[0]}.{cap[1]})")
        else:
            print(f"[!] Found GPU with capability {cap[0]}.{cap[1]} < 7.0. Using CPU.")
    except Exception as e:
        print(f"[!] CUDA check failed: {e}. Using CPU.")
elif torch.backends.mps.is_available():
    device = torch.device("mps")

print(f"[*] Training Device: {device} | Input Directory: {KAGGLE_INPUT}")

In [ ]:
# 2. Multilingual Clinical NLP Extractor with Regex Proximity & Negation
NEG_WORDS = ["no", "without", "free of", "negative for", "intact", "unremarkable", "sin", "sans", "pas de", "kein", "keine", "keinerlei", "bez", "nema", "yok", "χωρίς", "без"]
UNC_WORDS = ["possible", "suspected", "indeterminate", "equivocal", "questionable", "cannot rule out", "posible", "sospecha", "suspecté", "möglich", "verdacht auf", "vjerojatno"]

ONTOLOGY_TERMS = {
    "ACL": ["acl tear", "tear of the anterior cruciate ligament", "acl rupture", "complete acl disruption", "rotura del lca", "vkb-ruptur", "rupture du lca", "ruptura prednjeg križnog", "ön çapraz bağ rüptürü", "ρήξη πρόσθιου χιαστού"],
    "MCL": ["mcl tear", "tear of the medial collateral ligament", "mcl sprain", "rotura del lli", "innenbandruptur", "entorse du lcm", "ruptura mcl", "medial kollateral ligaman yırtık"],
    "Medial Meniscus": ["medial meniscus tear", "tear of the medial meniscus", "posterior horn medial meniscus", "rotura del menisco interno", "innenmeniskusruptur", "fissure du ménisque médial", "ruptura medijalnog meniska", "medial menisküs yırtık", "ρήξη έσω μηνίσκου"],
    "Lateral Meniscus": ["lateral meniscus tear", "tear of the lateral meniscus", "rotura del menisco externo", "außenmeniskusruptur", "fissure du ménisque latéral", "ruptura lateralnog meniska", "lateral menisküs yırtık", "ρήξη έξω μηνίσκου"],
    "Medial OA": ["medial compartment osteoarthritis", "medial oa", "oa of medial compartment", "oa of medial", "medial joint space narrowing", "gonartrosis medial", "mediale gonarthrose", "artrosis medial", "oa medijalnog", "gonartrotske promjene medijalnog"],
    "Lateral OA": ["lateral compartment osteoarthritis", "lateral oa", "oa of lateral compartment", "oa of lateral", "gonartrosis lateral", "laterale gonarthrose", "artrosis lateral", "oa lateralnog"],
    "PF OA": ["patellofemoral osteoarthritis", "pf oa", "oa patellofemoral", "chondromalacia patellae", "condropatía rotuliana", "artrosis femoropatelar", "retropatellare arthrose", "pf artrotske promjene", "hondromalacija patele"],
    "Effusion": ["joint effusion", "knee effusion", "suprapatellar effusion", "effusion with synovitis", "derrame articular", "gelenkerguss", "épanchement intra-articulaire", "zglobni izljev", "eklem efüzyonu", "ставεν izliv", "ставен излив"],
    "Synovitis": ["synovitis", "synovial hypertrophy", "synovial thickening", "sinovitis", "synovialisverdickung", "synovite", "sinovit", "синовит"],
    "Baker's": ["baker's cyst", "bakers cyst", "popliteal cyst", "quiste de baker", "baker-zyste", "kyste de baker", "bakerova cista", "baker kisti", "κύστη baker"],
    "Contusion": ["bone contusion", "bone bruise", "bone marrow edema", "contusión ósea", "knochenkontusion", "œdème médullaire osseux", "koštani edem", "kemik iliği ödemi", "οστικό μώλωπα"],
    "Fracture": ["fracture", "tibial plateau fracture", "patellar fracture", "avulsion fracture", "osteochondral fracture", "fractura", "fraktur", "prijelom", "kırık", "κάταγμα", "фрактура"]
}

def extract_study_report(text: str) -> Dict[str, Tuple[float, bool, float]]:
    text = str(text).lower()
    results = {}
    for target, terms in ONTOLOGY_TERMS.items():
        state = "not_mentioned"
        matched = False
        for t in terms:
            idx = text.find(t)
            if idx != -1:
                ctx = text[max(0, idx - 60):idx]
                if any(neg in ctx for neg in NEG_WORDS):
                    state = "negative"
                elif any(unc in ctx for unc in UNC_WORDS):
                    state = "uncertain"
                else:
                    state = "positive"
                matched = True
                break
        if state == "positive":
            results[target] = (0.95, True, 1.0) # (prob, mask, weight)
        elif state == "negative":
            results[target] = (0.05, True, 1.0)
        elif state == "uncertain":
            results[target] = (0.50, False, 0.0)
        else:
            results[target] = (0.10, False, 0.0)
    return results

In [ ]:
# 3. Robust DICOM Slice Geometry, Resizing & Dataset
def calculate_slice_position_along_normal(ds):
    try:
        if not hasattr(ds, "ImageOrientationPatient") or not hasattr(ds, "ImagePositionPatient"):
            return None, "unknown"
        iop = [float(x) for x in ds.ImageOrientationPatient]
        ipp = [float(x) for x in ds.ImagePositionPatient]
        if len(iop) != 6 or len(ipp) != 3:
            return None, "unknown"
        r, c = np.array(iop[:3], dtype=np.float64), np.array(iop[3:], dtype=np.float64)
        normal = np.cross(r, c)
        norm = np.linalg.norm(normal)
        if norm < 1e-6:
            return None, "unknown"
        unit_normal = normal / norm
        pos = float(np.dot(np.array(ipp, dtype=np.float64), unit_normal))
        d_axis = int(np.argmax(np.abs(unit_normal)))
        plane = ["sagittal", "coronal", "axial"][d_axis] if d_axis < 3 else "unknown"
        return pos, plane
    except Exception:
        return None, "unknown"

def sort_study_dicoms_by_plane(dicom_files: List[Path]) -> Dict[str, List[Path]]:
    plane_files = {"sagittal": [], "coronal": [], "axial": [], "unknown": []}
    for f in dicom_files:
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True)
            pos, plane = calculate_slice_position_along_normal(ds)
            if pos is None:
                pos = float(getattr(ds, "InstanceNumber", 0))
            plane_files[plane].append((f, pos))
        except Exception:
            plane_files["unknown"].append((f, 0.0))
    for p in plane_files:
        plane_files[p].sort(key=lambda x: x[1])
    return {p: [x[0] for x in plane_files[p]] for p in plane_files}

def extract_study_metadata(dicom_files: List[Path]) -> np.ndarray:
    meta = np.zeros(16, dtype=np.float32)
    if not dicom_files:
        return meta
    try:
        ds = pydicom.dcmread(str(dicom_files[0]), stop_before_pixels=True)
        sex = str(getattr(ds, "PatientSex", "")).upper()
        meta[0] = 1.0 if sex == "M" else (0.0 if sex == "F" else 0.5)
        field = float(getattr(ds, "MagneticFieldStrength", 1.5) or 1.5)
        meta[1] = 1.0 if field >= 2.5 else 0.0
        meta[2] = min(field / 3.0, 1.0)
        manuf = str(getattr(ds, "Manufacturer", "")).lower()
        meta[3] = 1.0 if "siemens" in manuf else 0.0
        meta[4] = 1.0 if "ge" in manuf or "general electric" in manuf else 0.0
        meta[5] = 1.0 if "philips" in manuf else 0.0
        meta[6] = min(len(dicom_files) / 150.0, 1.0)
    except Exception:
        pass
    return meta

def load_and_resize_slice(fp: Path, target_size: int = 224) -> Optional[np.ndarray]:
    try:
        ds = pydicom.dcmread(str(fp))
        arr = ds.pixel_array.astype(np.float32)
        vmin, vmax = np.percentile(arr, 0.5), np.percentile(arr, 99.5)
        vmax = vmax if vmax > vmin else vmin + 1.0
        arr = np.clip((arr - vmin) / (vmax - vmin), 0.0, 1.0)
        t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)
        if t.shape[-1] != target_size or t.shape[-2] != target_size:
            t = F.interpolate(t, size=(target_size, target_size), mode="bilinear", align_corners=False)
        return t.squeeze(0).squeeze(0).numpy()
    except Exception:
        return None

class TriPlaneDataset(Dataset):
    def __init__(self, df, data_dir, image_size=224, slices_per_plane=12, is_training=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.image_size = image_size
        self.slices = slices_per_plane
        self.is_training = is_training

    def __len__(self):
        return len(self.df)

    def _load_plane(self, files):
        if not files:
            return np.zeros((self.slices, 3, self.image_size, self.image_size), dtype=np.float32)
        slices = []
        for f in files:
            s_arr = load_and_resize_slice(f, target_size=self.image_size)
            if s_arr is not None:
                slices.append(s_arr)
        if not slices:
            return np.zeros((self.slices, 3, self.image_size, self.image_size), dtype=np.float32)
        vol = np.stack(slices, axis=0) # (Z, image_size, image_size)
        Z = len(vol)
        idxs = np.linspace(0, Z - 1, self.slices).astype(int)
        sampled = []
        for i in idxs:
            triplet = [vol[np.clip(i + o, 0, Z - 1)] for o in [-1, 0, 1]]
            sampled.append(np.stack(triplet, axis=0))
        return np.stack(sampled, axis=0) # (S, 3, H, W)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid = str(row[ID_COLUMN])
        study_dir = self.data_dir / "train_series" / uid
        files = list(study_dir.glob("**/*.dcm")) if study_dir.exists() else []
        planes = sort_study_dicoms_by_plane(files) if files else {"sagittal": [], "coronal": [], "axial": []}
        meta = extract_study_metadata(files)

        sag_files = planes["sagittal"] if len(planes.get("sagittal", [])) > 0 else planes.get("unknown", [])
        cor_files = planes["coronal"] if len(planes.get("coronal", [])) > 0 else planes.get("unknown", [])
        ax_files = planes["axial"] if len(planes.get("axial", [])) > 0 else planes.get("unknown", [])

        sag = self._load_plane(sag_files)
        cor = self._load_plane(cor_files)
        ax = self._load_plane(ax_files)

        targets = np.zeros(len(TARGET_NAMES), dtype=np.float32)
        masks = np.zeros(len(TARGET_NAMES), dtype=bool)
        weights = np.ones(len(TARGET_NAMES), dtype=np.float32)

        for t_idx, t_name in enumerate(TARGET_NAMES):
            if f"{t_name}_prob" in row:
                targets[t_idx] = float(row[f"{t_name}_prob"])
                masks[t_idx] = bool(row.get(f"{t_name}_loss_mask", True))
                weights[t_idx] = float(row.get(f"{t_name}_loss_weight", 1.0))
            elif t_name in row and not pd.isna(row[t_name]):
                targets[t_idx] = float(row[t_name])
                masks[t_idx] = True
                weights[t_idx] = 5.0 # Gold expert weight

        return {
            "sagittal": torch.from_numpy(sag).float(),
            "coronal": torch.from_numpy(cor).float(),
            "axial": torch.from_numpy(ax).float(),
            "metadata": torch.from_numpy(meta).float(),
            "targets": torch.from_numpy(targets).float(),
            "loss_mask": torch.from_numpy(masks).bool(),
            "loss_weight": torch.from_numpy(weights).float(),
        }

In [ ]:
# 4. Multimodal HMIL Neural Network & Stabilized Asymmetric Loss
class TargetSpecificAttentionPooling(nn.Module):
    def __init__(self, in_features, num_targets=12, hidden_dim=128):
        super().__init__()
        self.num_targets = num_targets
        self.attention_nets = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_features, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, 1),
            ) for _ in range(num_targets)
        ])
    def forward(self, x):
        B, S, D = x.shape
        reps = []
        for k in range(self.num_targets):
            attn = F.softmax(self.attention_nets[k](x).squeeze(-1), dim=-1).unsqueeze(-1)
            reps.append(torch.sum(x * attn, dim=1))
        return torch.stack(reps, dim=1)

class MultimodalHMILModel(nn.Module):
    def __init__(self, num_targets=12, feature_dim=128, meta_dim=16):
        super().__init__()
        self.num_targets = num_targets
        self.feature_dim = feature_dim
        self.planes = ["sagittal", "coronal", "axial"]
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, feature_dim, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(feature_dim), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Linear(feature_dim, feature_dim),
        )
        self.plane_pools = nn.ModuleDict({
            p: TargetSpecificAttentionPooling(feature_dim, num_targets, hidden_dim=feature_dim) for p in self.planes
        })
        self.view_gates = nn.ModuleList([
            nn.Sequential(nn.Linear(feature_dim * 3, 64), nn.ReLU(inplace=True), nn.Linear(64, 3))
            for _ in range(num_targets)
        ])
        self.meta_net = nn.Sequential(nn.Linear(meta_dim, 32), nn.ReLU(inplace=True), nn.Linear(32, feature_dim))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(feature_dim * 2, 64), nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(64, 1))
            for _ in range(num_targets)
        ])

    def forward(self, sagittal, coronal, axial, metadata=None):
        plane_inputs = {"sagittal": sagittal, "coronal": coronal, "axial": axial}
        first_t = sagittal
        B, dev = first_t.shape[0], first_t.device
        plane_reps = {}
        for p in self.planes:
            x = plane_inputs[p]
            B_p, S, C, H, W = x.shape
            feats = self.stem(x.view(B_p * S, C, H, W)).view(B_p, S, self.feature_dim)
            plane_reps[p] = self.plane_pools[p](feats)
        
        stacked = torch.stack([plane_reps[p] for p in self.planes], dim=2)
        meta_emb = self.meta_net(metadata if metadata is not None else torch.zeros((B, 16), device=dev))
        logits = []
        for k in range(self.num_targets):
            k_cat = stacked[:, k, :, :].reshape(B, 3 * self.feature_dim)
            w = F.softmax(self.view_gates[k](k_cat), dim=-1).unsqueeze(-1)
            vis = torch.sum(stacked[:, k, :, :] * w, dim=1)
            logits.append(self.heads[k](torch.cat([vis, meta_emb], dim=-1)))
        return torch.cat(logits, dim=-1)

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.5, clip=0.05, eps=1e-7):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets, loss_mask=None, weights=None):
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1.0 - x_sigmoid
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)
        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps, max=1.0))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps, max=1.0))
        loss = los_pos + los_neg

        pt0 = xs_pos * targets
        pt1 = xs_neg * (1.0 - targets)
        pt = (pt0 + pt1).clamp(min=0.0, max=1.0)
        one_sided_gamma = self.gamma_pos * targets + self.gamma_neg * (1.0 - targets)
        one_sided_w = torch.pow(torch.clamp(1.0 - pt, min=0.0, max=1.0), one_sided_gamma)
        loss = -loss * one_sided_w

        if weights is not None:
            loss = loss * weights
        if loss_mask is not None:
            loss = loss * loss_mask.float()
            num_valid = torch.clamp(loss_mask.float().sum(), min=1.0)
            return loss.sum() / num_valid
        return loss.mean()

In [ ]:
# 5. Execute 5-Fold Training Loop & Compute Out-of-Fold Macro ROC-AUC
train_df = pd.read_csv(TRAIN_CSV)
print(f"[*] Loaded training set: {len(train_df)} studies")

# Extract report pseudo labels if not already columns
records = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Extracting NLP Supervision"):
    rec = {ID_COLUMN: row[ID_COLUMN]}
    has_expert = False
    for t in TARGET_NAMES:
        if t in row and not pd.isna(row[t]):
            rec[f"{t}_prob"] = float(row[t])
            rec[f"{t}_loss_mask"] = True
            rec[f"{t}_loss_weight"] = 5.0
            has_expert = True
    if not has_expert:
        extracted = extract_study_report(row.get("Report", ""))
        for t in TARGET_NAMES:
            prob, mask, weight = extracted[t]
            rec[f"{t}_prob"] = prob
            rec[f"{t}_loss_mask"] = mask
            rec[f"{t}_loss_weight"] = weight
    records.append(rec)

processed_df = pd.DataFrame(records)

# Stratified 5-Fold Split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
strat_col = (processed_df["ACL_prob"] > 0.5).astype(int) + (processed_df["Effusion_prob"] > 0.5).astype(int)*2
processed_df["fold"] = -1
for f, (tr_idx, val_idx) in enumerate(skf.split(processed_df, strat_col)):
    processed_df.loc[val_idx, "fold"] = f

EPOCHS = 10
BATCH_SIZE = 8
oof_predictions = np.zeros((len(processed_df), len(TARGET_NAMES)), dtype=np.float32)
oof_targets = np.zeros((len(processed_df), len(TARGET_NAMES)), dtype=np.float32)
oof_masks = np.zeros((len(processed_df), len(TARGET_NAMES)), dtype=bool)

for fold in range(5):
    print(f"\n{'='*60}\n         STARTING 5-FOLD TRAINING: FOLD {fold}/5\n{'='*60}")
    tr_sub = processed_df[processed_df["fold"] != fold].copy()
    val_sub = processed_df[processed_df["fold"] == fold].copy()
    val_indices = val_sub.index.values

    tr_ds = TriPlaneDataset(tr_sub, KAGGLE_INPUT, is_training=True)
    val_ds = TriPlaneDataset(val_sub, KAGGLE_INPUT, is_training=False)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = MultimodalHMILModel().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * max(1, len(tr_loader)), eta_min=1e-6)
    criterion = AsymmetricLoss()

    best_val_auc = 0.0
    best_val_preds = None

    for epoch in range(1, EPOCHS + 1):
        # Train
        model.train()
        tr_loss = 0.0
        for b in tqdm(tr_loader, desc=f"Fold {fold} Ep {epoch} [Train]", leave=False):
            sag, cor, ax = b["sagittal"].to(device), b["coronal"].to(device), b["axial"].to(device)
            meta, tgt = b["metadata"].to(device), b["targets"].to(device)
            mask, weight = b["loss_mask"].to(device), b["loss_weight"].to(device)

            optimizer.zero_grad()
            logits = model(sag, cor, ax, metadata=meta)
            loss = criterion(logits, tgt, loss_mask=mask, weights=weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            tr_loss += loss.item()
        tr_loss /= max(1, len(tr_loader))

        # Eval
        model.eval()
        val_loss = 0.0
        p_list, t_list, m_list = [], [], []
        with torch.no_grad():
            for b in tqdm(val_loader, desc=f"Fold {fold} Ep {epoch} [Val]", leave=False):
                sag, cor, ax = b["sagittal"].to(device), b["coronal"].to(device), b["axial"].to(device)
                meta, tgt = b["metadata"].to(device), b["targets"].to(device)
                mask = b["loss_mask"].to(device)
                logits = model(sag, cor, ax, metadata=meta)
                loss = criterion(logits, tgt, loss_mask=mask)
                val_loss += loss.item()
                p_list.append(torch.sigmoid(logits).cpu().numpy())
                t_list.append(tgt.cpu().numpy())
                m_list.append(mask.cpu().numpy())
        val_loss /= max(1, len(val_loader))

        y_p = np.vstack(p_list)
        y_t = np.vstack(t_list)
        y_m = np.vstack(m_list)

        aucs = []
        for k in range(len(TARGET_NAMES)):
            val_k = y_m[:, k]
            if np.unique(y_t[val_k, k]).size > 1:
                aucs.append(roc_auc_score(y_t[val_k, k], y_p[val_k, k]))
        val_macro_auc = float(np.mean(aucs)) if aucs else 0.0
        print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro AUC: {val_macro_auc:.4f}")

        if val_macro_auc > best_val_auc or epoch == 1:
            best_val_auc = val_macro_auc
            best_val_preds = y_p
            ckpt_path = CHECKPOINT_DIR / f"model_fold_{fold}_best.pt"
            torch.save({"model_state_dict": model.state_dict(), "fold": fold, "val_macro_auc": val_macro_auc}, ckpt_path)
            if Path("/kaggle/working").exists():
                torch.save({"model_state_dict": model.state_dict(), "fold": fold, "val_macro_auc": val_macro_auc}, Path(f"/kaggle/working/model_fold_{fold}_best.pt"))
            print(f"  [+] Saved new best fold {fold} checkpoint -> {ckpt_path}")

    oof_predictions[val_indices] = best_val_preds
    oof_targets[val_indices] = y_t
    oof_masks[val_indices] = y_m

# 6. Full 5-Fold OOF Evaluation Report & Parquet Export
print(f"\n{'='*60}\n       FINAL 5-FOLD OUT-OF-FOLD (OOF) PERFORMANCE REPORT\n{'='*60}")
oof_aucs = {}
for k, t_name in enumerate(TARGET_NAMES):
    m_k = oof_masks[:, k]
    if np.unique(oof_targets[m_k, k]).size > 1:
        t_auc = float(roc_auc_score(oof_targets[m_k, k], oof_predictions[m_k, k]))
        oof_aucs[t_name] = t_auc
        print(f"  {t_name:<30}: {t_auc:.4f} (Evaluated on {m_k.sum()} labeled validation studies)")
    else:
        print(f"  {t_name:<30}: [Single Class / Undefined]")

master_oof_macro = float(np.mean(list(oof_aucs.values())))
print("-"*60)
print(f" OVERALL 5-FOLD OOF MACRO ROC-AUC: {master_oof_macro:.4f}")
print(f"{'='*60}")

# Save OOF Parquet
oof_df = pd.DataFrame(oof_predictions, columns=[f"{t}_pred" for t in TARGET_NAMES])
oof_df.insert(0, ID_COLUMN, processed_df[ID_COLUMN])
oof_df["fold"] = processed_df["fold"]
for k, t in enumerate(TARGET_NAMES):
    oof_df[f"{t}_target"] = oof_targets[:, k]
    oof_df[f"{t}_loss_mask"] = oof_masks[:, k]
oof_path = Path("/kaggle/working/oof_predictions_v1.parquet") if Path("/kaggle/working").exists() else Path("experiments/oof_predictions_v1.parquet")
oof_df.to_parquet(oof_path, index=False)
print(f"[+] Exported OOF predictions -> {oof_path}")